#### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# model = init_chat_model("groq:qwen/qwen3.6-27b")
# response = model.invoke("Why do parrots talk?")

### Summarization Middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.


In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

## Message based summarization
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]

)

In [7]:
## run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [11]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")
 

Messages: {'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze the Request:**\n   - **Role:** Context Extraction Assistant\n   - **Objective:** Extract the highest quality/most relevant context from the provided conversation history to replace it and free up tokens.\n   - **Format:** Must use specific sections: `## SESSION INTENT`, `## SUMMARY`, `## ARTIFACTS`, `## NEXT STEPS`. Each must be populated or marked "None".\n   - **Constraint:** Respond ONLY with the extracted context. No extra text.\n   - **Input Conversation:** A short sequence of math questions and answers.\n     - Human: "What is 15-7?" -> AI: 15 - 7 equals **8**.\n     - Human: "What is 3*3?" -> AI: 3 × 3 equals **9**.\n     - Human: "What is 4*4?" -> AI: 4 × 4 equals **16**.\n     - Human: "What is 2+2?" (Unanswered, pending)\n   - **Pattern/Style:** The AI consistently responds with the format "[Number] [Operation Symbol] [Number] e

#### token size

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [13]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~118 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='7207201e-7c81-40ea-9f51-6b221b11675d'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to find hotels in Paris.\n2.  **Identify Available Tools**: I have a `search_hotels` function that takes a `city` parameter.\n3.  **Check Parameters**: The function requires `city` (string). The user provided "Paris".\n4.  **Construct Tool Call**: `search_hotels(city="Paris")`\n5.  **Execute Tool Call**: Call the function.\n6.  **Process Response**: Format the output based on the function\'s response. (I will simulate the tool call now). \nWait, I need to actually generate the tool call. I will output the function call.✅\n', 'tool_calls': [{'id': 'pbq3h3b07', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_

### Fraction

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq


@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"


# Create the model explicitly and provide its context window
model = ChatGroq(
    model="qwen/qwen3.6-27b",
    profile={
        "max_input_tokens": 128000
    }
)


agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,

            # Fraction of the model's maximum input context
            trigger=("fraction", 0.005),
            keep=("fraction", 0.002),
        ),
    ],
)


config = {
    "configurable": {
        "thread_id": "test-1"
    }
}


def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4


cities = [
    "Paris",
    "London",
    "Tokyo",
    "New York",
    "Dubai",
    "Singapore"
]


for city in cities:
    response = agent.invoke(
        {
            "messages": [
                HumanMessage(content=f"Hotels in {city}")
            ]
        },
        config=config
    )

    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000

    print(
        f"{city}: ~{tokens} tokens "
        f"({fraction:.4%}), "
        f"{len(response['messages'])} msgs"
    )

Paris: ~48 tokens (0.0375%), 4 msgs
London: ~98 tokens (0.0766%), 8 msgs
Tokyo: ~147 tokens (0.1148%), 12 msgs
New York: ~1404 tokens (1.0969%), 12 msgs
Dubai: ~1312 tokens (1.0250%), 12 msgs
Singapore: ~1719 tokens (1.3430%), 12 msgs


### Human In the Loop MiddleWare
Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [17]:
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [18]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [19]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='932a3e35-d15a-4fc2-bc28-c2747823e2e2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: john@test.com\n   - Subject: Hello\n   - Body: How are you?\n2.  **Identify Tool**: `send_email_tool` matches the requirement.\n3.  **Check Parameters**:\n   - `recipient`: "john@test.com"\n   - `subject`: "Hello"\n   - `body`: "How are you?"\n   All required parameters are provided.\n4.  **Execute Tool**: Call `send_email_tool` with the specified parameters.\n5.  **Formulate Response**: Return the result of the tool execution. (Mock tool will likely return success/status). I will just call the tool.✅\n   No extra steps needed. Proceed. \n   Tool call: `send_email_tool(recipient="john@test.com", subject="Hello", body="How are you?")`✅\

#### Approve

In [20]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: The email has been sent to john@test.com with the subject 'Hello'.


In [22]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='932a3e35-d15a-4fc2-bc28-c2747823e2e2'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: john@test.com\n   - Subject: Hello\n   - Body: How are you?\n2.  **Identify Tool**: `send_email_tool` matches the requirement.\n3.  **Check Parameters**:\n   - `recipient`: "john@test.com"\n   - `subject`: "Hello"\n   - `body`: "How are you?"\n   All required parameters are provided.\n4.  **Execute Tool**: Call `send_email_tool` with the specified parameters.\n5.  **Formulate Response**: Return the result of the tool execution. (Mock tool will likely return success/status). I will just call the tool.✅\n   No extra steps needed. Proceed. \n   Tool call: `send_email_tool(recipient="john@test.com", subject="Hello", body="How are you?")`✅\

#### Reject

In [23]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [24]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [25]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='58f0da3f-edd6-469f-a05c-4c3b4e208c05'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: john@test.com\n   - Subject: \'Hello\'\n   - Body: \'How are you?\'\n2.  **Identify Available Tools**: `send_email_tool` matches the requirement.\n   - Parameters: `recipient` (string), `subject` (string), `body` (string)\n   - All required parameters are provided.\n3.  **Construct Tool Call**:\n   - `recipient`: "john@test.com"\n   - `subject`: "Hello"\n   - `body`: "How are you?"\n4.  **Execute Tool Call**: Call `send_email_tool` with the specified parameters.\n5.  **Return Response**: After tool execution, confirm the email was sent. (I will just output the tool call now).✅\n', 'tool_calls': [{'id': '6yf00h0s5', 'function': {'argume

In [26]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: It appears the action to send the email was rejected, so the message was not sent to john@test.com. Please let me know if you would like to attempt this again or if there is anything else I can assist you with.


In [27]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='58f0da3f-edd6-469f-a05c-4c3b4e208c05'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Analyze User Input**: User wants to send an email.\n   - Recipient: john@test.com\n   - Subject: \'Hello\'\n   - Body: \'How are you?\'\n2.  **Identify Available Tools**: `send_email_tool` matches the requirement.\n   - Parameters: `recipient` (string), `subject` (string), `body` (string)\n   - All required parameters are provided.\n3.  **Construct Tool Call**:\n   - `recipient`: "john@test.com"\n   - `subject`: "Hello"\n   - `body`: "How are you?"\n4.  **Execute Tool Call**: Call `send_email_tool` with the specified parameters.\n5.  **Return Response**: After tool execution, confirm the email was sent. (I will just output the tool call now).✅\n', 'tool_calls': [{'id': '6yf00h0s5', 'function': {'argume

#### Edit

In [28]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [30]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

In [31]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='6a425bd1-1e8e-420c-b615-59209c1f639b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI need to use the `send_email_tool` function.\nThe recipient is 'wrong@email.com'.\nThe subject is 'Test'.\nThe body is 'Hello'.\nAll required parameters are provided. I will call the function.\n", 'tool_calls': [{'id': 'x1zxzxx9r', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 372, 'total_tokens': 486, 'completion_time': 0.223168407, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.032620564, 'prompt_tokens_details': None, 'queue_time': 0.047269819, 'total_time': 0.255788971}, 'model_name': 'qwen/qwen3.

In [32]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "correct@email.com",
                                "subject": "Corrected Subject",
                                "body": "This was edited by human before sending"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: The email was successfully sent to correct@email.com with the subject 'Corrected Subject' and the body content 'This was edited by human before sending'.


In [33]:
result

{'messages': [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='6a425bd1-1e8e-420c-b615-59209c1f639b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants to send an email.\nI need to use the `send_email_tool` function.\nThe recipient is 'wrong@email.com'.\nThe subject is 'Test'.\nThe body is 'Hello'.\nAll required parameters are provided. I will call the function.\n", 'tool_calls': [{'id': 'x1zxzxx9r', 'function': {'arguments': '{"body":"Hello","recipient":"wrong@email.com","subject":"Test"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 372, 'total_tokens': 486, 'completion_time': 0.223168407, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.032620564, 'prompt_tokens_details': None, 'queue_time': 0.047269819, 'total_time': 0.255788971}, 'model_name': 'qwen/qwen3.

#### Explore more middleware from documentation

https://reference.langchain.com/python/langchain/middleware